# 04 - Model Evaluation
In-depth evaluation of the trained models.

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.data_loader import load_processed_data
from src.feature_engineering import get_feature_target_split, split_data, scale_age
from src.models import get_models, load_model
from src.evaluation import evaluate_model, cross_validate, compare_models, get_best_model

%matplotlib inline

In [ ]:
df = load_processed_data()
df, scaler = scale_age(df)
X, y = get_feature_target_split(df, target='treatment')
X_train, X_test, y_train, y_test = split_data(X, y)

In [ ]:
models = get_models()
results = compare_models(models, X_train, y_train, X_test, y_test)

## Accuracy Comparison

In [ ]:
names = list(results.keys())
accuracies = [results[n]['test_metrics']['accuracy'] for n in names]
cv_means = [results[n]['cv_mean'] for n in names]

fig, ax = plt.subplots(figsize=(12, 6))
x = np.arange(len(names))
width = 0.35
ax.bar(x - width/2, accuracies, width, label='Test Accuracy')
ax.bar(x + width/2, cv_means, width, label='CV Mean Accuracy')
ax.set_ylabel('Accuracy')
ax.set_title('Model Comparison')
ax.set_xticks(x)
ax.set_xticklabels(names, rotation=45, ha='right')
ax.legend()
plt.tight_layout()
plt.show()

## Best Model - Confusion Matrix

In [ ]:
best_name, best_model = get_best_model(results)
metrics = evaluate_model(best_model, X_test, y_test)

print(f'Best Model: {best_name}')
print(f'\nClassification Report:\n{metrics["classification_report"]}')

plt.figure(figsize=(6, 5))
sns.heatmap(metrics['confusion_matrix'], annot=True, fmt='d', cmap='Blues')
plt.title(f'Confusion Matrix - {best_name}')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.show()

## All Models - Classification Reports

In [ ]:
for name, res in results.items():
    print(f'=== {name} ===')
    print(f'Test Accuracy: {res["test_metrics"]["accuracy"]:.4f}')
    print(f'CV Mean: {res["cv_mean"]:.4f} (+/- {res["cv_std"]:.4f})')
    print(res['test_metrics']['classification_report'])
    print()